# Maintenance Work Order Data Workflow

**Student:** Peyman Mohammad Hassan  
**Dataset:** Synthetic CMMS Maintenance Work Orders

This notebook develops a reproducible Python data workflow for a synthetic CMMS-style maintenance dataset. The records are simulated for educational purposes and do not represent real customer, technician, property, or asset data.

The project focuses on data ingestion, cleaning, exploratory analysis, visualization, and interpretation as a foundation for later machine-learning work-order priority prediction.


## 1. Setup

Import the required Python libraries and make the repository root available on the Python import path.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import RAW_DATA_FILE, PROCESSED_DATA_FILE
from src.generate_data import save_dataset
from src.cleaning import standardize_categories, remove_duplicate_work_orders, flag_invalid_values, impute_missing_values
from src.eda import priority_summary, asset_type_summary
from src.visualizations import plot_priority_distribution, plot_failures_by_priority, plot_median_cost_by_asset_type, plot_resolution_by_priority

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Data Ingestion

The raw CSV is generated reproducibly when it is not already present. The dataset is then loaded with Pandas and the first rows are displayed.


In [ ]:
if not RAW_DATA_FILE.exists():
    save_dataset()

df_raw = pd.read_csv(RAW_DATA_FILE, parse_dates=["created_date"], keep_default_na=False, na_values=[""])
print(f"Loaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.")
df_raw.head()


## 3. Data Cleaning

The raw dataset intentionally contains missing values, inconsistent categorical formatting, duplicate work orders, and invalid values. The documented functions below make those cleaning steps explicit and reproducible.


In [ ]:
def clean_categories(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize inconsistent categorical values in the maintenance dataset."""
    return standardize_categories(df)


def clean_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate maintenance work orders using work_order_id."""
    return remove_duplicate_work_orders(df)


def clean_invalid_and_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Handle documented invalid values and impute selected missing values."""
    return impute_missing_values(flag_invalid_values(df))

df_clean = clean_categories(df_raw)
df_clean = clean_duplicates(df_clean)
df_clean = clean_invalid_and_missing(df_clean)
PROCESSED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DATA_FILE, index=False)
print(f"Raw rows: {len(df_raw):,}; Clean rows: {len(df_clean):,}")
df_clean.head()


Cleaning was needed because inconsistent category labels could fragment groups, duplicate work orders could double-count events, and injected invalid values could distort summaries. Missing values are handled explicitly rather than silently discarded, although imputation can reduce variability and should be treated as an analytical assumption.


## 4. Exploratory Data Analysis


In [ ]:
def exploratory_summary(df: pd.DataFrame) -> dict:
    """Return reusable priority, asset-type, and numeric summaries for EDA."""
    return {
        "priority_summary": priority_summary(df),
        "asset_type_summary": asset_type_summary(df),
        "numeric_summary": df.select_dtypes(include=np.number).describe().round(2),
    }

eda_results = exploratory_summary(df_clean)
display(eda_results["priority_summary"])
display(eda_results["asset_type_summary"])
display(eda_results["numeric_summary"])


The EDA shows that Medium work orders are the largest priority group while Emergency work orders are intentionally less common. Asset categories also differ in volume, maintenance history, and cost, which makes them useful for descriptive comparison.


## 5. Visualizations


In [ ]:
plot_priority_distribution(df_clean)
plt.show()


**Figure 1:** Medium is the most common priority while Emergency is the least common, demonstrating an intentionally imbalanced target distribution.


In [ ]:
plot_failures_by_priority(df_clean)
plt.show()


**Figure 2:** Average previous-failure counts vary by priority, suggesting maintenance history is associated with severity in the synthetic data.


In [ ]:
plot_median_cost_by_asset_type(df_clean)
plt.show()


**Figure 3:** Median estimated repair cost differs by asset type, showing why categorical asset information is useful during exploratory analysis.


In [ ]:
plot_resolution_by_priority(df_clean)
plt.show()


**Figure 4:** Resolution-time distributions overlap across priority levels. Resolution time is useful descriptively but occurs after work completion and should not be used later as an intake-time predictor.


## 6. Summary and Interpretation

This workflow transforms a deliberately imperfect synthetic CMMS dataset into a cleaned dataset suitable for exploratory analysis. The analysis highlights an imbalanced priority distribution, differences across asset types, and associations involving maintenance history and cost.

The main assumptions are that the synthetic generator's thresholds define valid and invalid values for this educational dataset. The main limitation is that all records are simulated, so the patterns should not be generalized to real maintenance organizations without external validation. A surprising and useful finding is that some descriptive variables, such as resolution time, are informative for analysis but are not appropriate for future intake-time prediction because they are known only after the work is completed.
